# 🦙 Meta-Llama-3-8B Fine-Tuning on Google Colab (Free T4 GPU)
### Task: High-Performance Romanized Hinglish-to-English Neural Machine Translation

This notebook fine-tunes **Meta's flagship Llama-3 (8 Billion parameters)** on the custom web-scraped social media corpus using **4-bit QLoRA** (`bitsandbytes` NF4).

#### Why Meta-Llama-3-8B?
- **8 Billion Parameters**: Massive reasoning and cross-lingual capacity compared to 2B and 300M models.
- **128k Tokenizer Vocabulary**: Superior byte-level and subword compression for Romanized Indic words.
- **Engineered to Outperform SOTA Baselines**: Designed to exceed published benchmarks (RCMT @ 14.00 BLEU in LREC-COLING 2024, PACMANtrans @ 18.66 BLEU, and Google mT5 @ 19.94 BLEU).

#### Hardware & Execution Profile:
- **Google Colab Free T4 GPU** (~15.3 GB VRAM)
- **VRAM Footprint**: ~7.5 to 8.5 GB (Zero OOM risk via Paged AdamW 8-bit & gradient checkpointing)
- **Training Time**: ~45–55 minutes for 3 epochs

## Step 1: Verify Free T4 GPU Runtime
Ensure your Colab runtime is set to **GPU** (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi

## Step 2: Clone or Update Repository
Pulls the latest fine-tuning scripts and parallel dataset corpora.

In [ ]:
import os
if not os.path.exists("/content/NLP"):
    !git clone https://github.com/Nickhasntlost/NLP.git /content/NLP
    %cd /content/NLP
else:
    %cd /content/NLP
    !git pull origin main

## Step 3: Install Required Dependencies
Removes conflicting packages and installs compatible libraries for 4-bit QLoRA and evaluation metrics while preserving Colab's native CUDA PyTorch.

In [ ]:
# 1. Remove conflicting pre-installed packages
!pip uninstall -y torchao

# 2. Install compatible ML & evaluation libraries
!pip install -q -U transformers peft accelerate bitsandbytes sacrebleu evaluate rouge_score bert_score nltk

## Step 4: Hugging Face Authentication (Required for Meta-Llama-3)
Meta-Llama-3-8B is a gated model. You must:
1. Request access on Hugging Face: [https://huggingface.co/meta-llama/Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B)
2. Copy your User Access Token from: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Set your token as a secret named `HF_TOKEN` in Colab's left sidebar (🔑 icon) with **Notebook access ON**, or enter it interactively.

In [ ]:
import os
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if hf_token and str(hf_token).strip():
    token_str = str(hf_token).strip()
    os.environ['HF_TOKEN'] = token_str
    from huggingface_hub import login
    login(token=token_str)
    print("✅ Authenticated successfully using Colab HF_TOKEN secret!")
else:
    print("⚠️ HF_TOKEN secret not found or notebook access toggle is off. Please log in interactively:")
    from huggingface_hub import login
    login()

## Step 5: Run 4-Bit QLoRA Fine-Tuning
Fine-tunes `meta-llama/Meta-Llama-3-8B` using:
- 4-bit NormalFloat (NF4) quantization + double quantization
- LoRA rank $r=32$, $\alpha=64$ across all 7 linear projection layers (`q, k, v, o, gate, up, down`)
- Strict loss masking on prompt tokens (gradient updates only on English translation)
- Memory-optimized for T4: Gradient Checkpointing + Paged AdamW 8-bit
- 3 Epochs on curated Hinglish-English parallel pairs

In [ ]:
!python finetune/finetune_llama3.py \
    --base_model meta-llama/Meta-Llama-3-8B \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 2e-4 \
    --lora_r 32 \
    --max_length 192

## Step 6: Multi-Metric Benchmark Evaluation
Evaluates the newly trained Llama-3 adapter on the 150-sample blind validation set and compares against published benchmarks (RCMT LREC-COLING 2024, PACMANtrans ICON 2023).

In [ ]:
!python finetune/evaluate_colab.py --model llama3 --limit 150

## Step 7: Interactive Translation Test
Test your fine-tuned Llama-3 model on representative Hinglish test sentences.

In [ ]:
test_sentences = [
    "bhaaaai kyaaa scene hai kal ka? plzz batao",
    "Aaj ka din bht zyada thk gya hu bhai",
    "Sir aap bahut achha padhate ho, mujhe samajh aa gaya.",
    "koi nhi bolega scripted h ye video",
    "Maine 636 tiktok videos khud report kiye the",
    "Yaar main rice khate hue dekh rahi thi aur mera mood kharab ho gaya."
]

for s in test_sentences:
    print(f"\n{'='*60}\nInput: {s}")
    !python finetune/translate_llama3.py --text "{s}"

## Step 8: Package LoRA Adapter & Download
Zips the trained adapter weights (~120 MB) so you can download them directly to your laptop or upload them to Hugging Face.

In [ ]:
import os
import shutil
from google.colab import files

adapter_dir = 'finetune/outputs/llama3_hinglish_lora'
if os.path.exists(adapter_dir):
    shutil.make_archive('llama3_hinglish_lora', 'zip', adapter_dir)
    print("✅ LoRA adapter packaged as: llama3_hinglish_lora.zip")
    files.download('llama3_hinglish_lora.zip')
else:
    print(f"⚠️ Adapter directory not found at {adapter_dir}. Please verify Step 5 completed.")